# News Category Classification — Main Notebook

Multi-class news classifier (42 categories) using headline + short description.

**Pipeline:** Kaggle dataset → preprocessing → TF-IDF + numeric features → 6 classical models + RoBERTa-embedding SVM → Gradio UI with Groq LLM explanations and similarity retrieval.

Per ADR-009, all pipeline code lives in this notebook. The only Python file maintained outside is `src/preprocessing.py` (because its `clean_text()` is unit-tested in CI).

Section headers map directly to sprint-1 tasks; later sprints fill in the remaining models and the full Gradio UI.

## Setup & Imports

Runs once at the top of the notebook. Idempotent — safe to re-run.

In [ ]:
import logging
import os
import sys
from pathlib import Path

# Make `src/` importable when running from the notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load .env if present (local Windows / Linux). On Colab use Colab Secrets UI instead.
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    pass

# Standard project paths.
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EMBEDDINGS_DIR = DATA_DIR / "embeddings"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFUSION_DIR = REPORTS_DIR / "confusion"
for d in (RAW_DIR, PROCESSED_DIR, EMBEDDINGS_DIR, MODELS_DIR, REPORTS_DIR, CONFUSION_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Single source of truth for randomness (ADR-section: reproducibility).
RANDOM_STATE = 42

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("main")
log.info(f"Project root: {PROJECT_ROOT}")

## 1. Data Loading — S1-T5

Download the News Category Dataset (~209K rows) from Kaggle. The first run hits Kaggle; subsequent runs read from local cache under `data/raw/`.

In [ ]:
"""S1-T5 — Kaggle download with local cache + JSON-Lines parse.

Returns a DataFrame with columns: link, headline, category,
short_description, authors, date. About 209K rows (full file ~80 MB).
First call hits Kaggle; subsequent calls read from `data/raw/`.
"""

import json

import pandas as pd

DATASET_SLUG = "rmisra/news-category-dataset"
DATASET_FILENAME = "News_Category_Dataset_v3.json"


def load_dataset(force_download: bool = False) -> pd.DataFrame:
    cache_path = RAW_DIR / DATASET_FILENAME
    if cache_path.exists() and not force_download:
        log.info(f"Using cached dataset at {cache_path} ({cache_path.stat().st_size / 1e6:.1f} MB)")
    else:
        # KaggleApi reads credentials from KAGGLE_CONFIG_DIR/kaggle.json.
        os.environ["KAGGLE_CONFIG_DIR"] = str(PROJECT_ROOT)
        from kaggle.api.kaggle_api_extended import KaggleApi

        api = KaggleApi()
        api.authenticate()
        log.info(f"Downloading {DATASET_SLUG} from Kaggle...")
        api.dataset_download_files(DATASET_SLUG, path=str(RAW_DIR), unzip=True)
        log.info(f"Cached to {cache_path}")

    rows = []
    with cache_path.open("r", encoding="utf-8") as fh:
        for line in fh:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    log.info(f"Loaded {len(df):,} articles | columns: {list(df.columns)}")
    return df


raw_df = load_dataset()
raw_df.head()

## 2. Preprocessing — S1-T6

Calls `clean_text()` from `src/preprocessing.py` (kept as a module so it can be unit-tested in CI). Adds three numeric features (word / char / punct counts) and writes the cleaned dataset to `data/processed/cleaned.parquet`.

In [ ]:
"""S1-T6 — Apply clean_text + numeric features over the dataset.

Reads `raw_df` from the cell above, produces the cleaned dataset with
the cleaned text column plus three numeric features
(word_count, char_count, punct_count), and persists to
`data/processed/cleaned.parquet`. Cached on subsequent runs.
"""

from tqdm.auto import tqdm

from src.preprocessing import build_numeric_features, clean_text

CLEANED_PATH = PROCESSED_DIR / "cleaned.parquet"

if CLEANED_PATH.exists():
    log.info(f"Using cached cleaned dataset at {CLEANED_PATH}")
    cleaned_df = pd.read_parquet(CLEANED_PATH)
else:
    log.info(f"Cleaning {len(raw_df):,} articles (~3-5 min on a laptop CPU)...")
    combined = (raw_df["headline"].fillna("") + " " + raw_df["short_description"].fillna("")).tolist()

    feats = [build_numeric_features(t) for t in tqdm(combined, desc="numeric")]
    texts = [clean_text(t) for t in tqdm(combined, desc="clean")]

    cleaned_df = pd.DataFrame(
        {
            "text": texts,
            "category": raw_df["category"].values,
            "word_count": [f["word_count"] for f in feats],
            "char_count": [f["char_count"] for f in feats],
            "punct_count": [f["punct_count"] for f in feats],
        }
    )
    cleaned_df.to_parquet(CLEANED_PATH, compression="snappy")
    log.info(f"Saved cleaned dataset to {CLEANED_PATH}")

log.info(f"Cleaned dataset: {len(cleaned_df):,} rows x {cleaned_df.shape[1]} columns")
cleaned_df.head()

## 3. EDA

Exploratory analysis lives in `notebooks/01_eda.ipynb`. Sprint 1 produces part 1 (shape / missing / class distribution); sprint 2 adds the full chart set.

## 4. Classical Features — S1-T8

TF-IDF (uni+bi-grams, capped at 50K features, sublinear TF) stacked with three numeric features (word / char / punct counts, scaled). Persisted to `models/`.

In [ ]:
"""S1-T8 — TF-IDF (uni+bi-gram, 50K cap, sublinear, min_df=2) + numeric features.

Stratified 80/20 split with `random_state=RANDOM_STATE`, vectorizer fit on
TRAIN ONLY (PRD F4), scaler fit on TRAIN ONLY. Persists vectorizer,
numeric scaler and label encoder to `models/`. Includes the round-trip
test (fit → save → load → transform produces an identical matrix).

Outputs in scope: X_train, X_test, y_train, y_test, plus the raw text
arrays for downstream display. Subsequent cells (S1-T9 train, S1-T10
evaluate) reuse these.
"""

import joblib
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Encode labels -> integer ids; persist for downstream inference.
label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(cleaned_df["category"].values)
joblib.dump(label_encoder, MODELS_DIR / "label_encoder.joblib")
log.info(f"Label encoder fit on {len(label_encoder.classes_)} classes; saved.")

# Stratified 80/20 split — same indices reused for every model in sprint 2.
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    cleaned_df["text"].values,
    cleaned_df[["word_count", "char_count", "punct_count"]].to_numpy(),
    y_all,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_all,
)
log.info(f"Train: {len(X_text_train):,}   Test: {len(X_text_test):,}")

# Fit TF-IDF on TRAIN only, transform both.
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,
    min_df=2,
)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)
joblib.dump(tfidf, MODELS_DIR / "tfidf_vectorizer.joblib")
log.info(f"TF-IDF vocab: {len(tfidf.vocabulary_):,} features")

# Fit StandardScaler on TRAIN numeric only, transform both.
scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)
joblib.dump(scaler, MODELS_DIR / "numeric_scaler.joblib")

# Sparse-stack TF-IDF + scaled numeric (3 cols).
X_train = hstack([X_tfidf_train, csr_matrix(X_num_train_scaled)]).tocsr()
X_test = hstack([X_tfidf_test, csr_matrix(X_num_test_scaled)]).tocsr()
log.info(f"X_train: {X_train.shape}   X_test: {X_test.shape}   nnz/row: {X_train.nnz / X_train.shape[0]:.1f}")

# Round-trip test (S1-T8 acceptance).
tfidf_reload = joblib.load(MODELS_DIR / "tfidf_vectorizer.joblib")
scaler_reload = joblib.load(MODELS_DIR / "numeric_scaler.joblib")
sample_idx = slice(0, 5)
roundtrip = hstack(
    [tfidf_reload.transform(X_text_test[sample_idx]), csr_matrix(scaler_reload.transform(X_num_test[sample_idx]))]
).tocsr()
diff = X_test[sample_idx] - roundtrip
assert diff.nnz == 0, f"Round-trip mismatch: {diff.nnz} non-zero deltas"
log.info("Round-trip test passed: load(saved).transform == original.transform on 5 sample rows")

## 5. Train Logistic Regression Baseline — S1-T9

Stratified 80/20 split, `class_weight='balanced'`, `GridSearchCV` over `C ∈ {0.1, 1, 10}`. Persists best estimator to `models/logreg_best.joblib`.

In [ ]:
"""S1-T9 — LogReg baseline with GridSearchCV.

Logistic Regression with `class_weight='balanced'`, multinomial via the
`saga` solver (good for large sparse text features), grid search over
`C ∈ {0.1, 1, 10}` with 3-fold CV scoring on macro-F1.

`saga` chosen over `lbfgs` for memory/speed on sparse 50K-dim input;
over `liblinear` because we want multinomial probabilities, not OvR.
Persists the best refit-on-full-train estimator to
`models/logreg_best.joblib`.
"""

import time

import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV

base = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    max_iter=300,
    random_state=RANDOM_STATE,
    n_jobs=1,  # let GridSearchCV parallelise across folds/params
)

search = GridSearchCV(
    base,
    param_grid={"C": [0.1, 1.0, 10.0]},
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)

t0 = time.time()
log.info("Starting GridSearchCV(cv=3) over C in {0.1, 1.0, 10.0} ...")
search.fit(X_train, y_train)
log.info(f"Done in {(time.time() - t0) / 60:.1f} min")
log.info(f"Best C: {search.best_params_['C']}, mean CV macro-F1: {search.best_score_:.4f}")

logreg_best = search.best_estimator_
joblib.dump(logreg_best, MODELS_DIR / "logreg_best.joblib")

# Quick test-set sanity check (full evaluation suite runs in S1-T10).
y_pred = logreg_best.predict(X_test)
test_macro_f1 = f1_score(y_test, y_pred, average="macro")
test_weighted_f1 = f1_score(y_test, y_pred, average="weighted")
log.info(f"Test set: macro-F1 = {test_macro_f1:.4f} | weighted-F1 = {test_weighted_f1:.4f}")
assert test_macro_f1 >= 0.45, f"S1-T9 acceptance fail: macro-F1 {test_macro_f1:.4f} < 0.45"
log.info("S1-T9 acceptance met (macro-F1 >= 0.45).")

## 6. Evaluate Models — S1-T10

Computes (accuracy, precision, recall, F1-macro, F1-weighted, ROC-AUC OvR) for each persisted model and appends a row to `reports/model_comparison.csv`. Saves a confusion-matrix PNG to `reports/confusion/<name>.png`.

In [ ]:
"""S1-T10 — `evaluate_model()` skeleton + first row in comparison table.

Computes (accuracy, precision_macro, recall_macro, f1_macro, f1_weighted,
roc_auc_ovr_macro) for any fitted sklearn classifier. Appends a row to
`reports/model_comparison.csv` (replacing any prior row for the same model)
and saves a row-normalised confusion-matrix PNG to
`reports/confusion/<name>.png`.

Sprint 2's models call this same function — that's why the per-model
row-replacement logic exists.
"""

from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

COMPARISON_CSV = REPORTS_DIR / "model_comparison.csv"


def evaluate_model(model, name, X_test, y_test, label_encoder=None):
    """Compute metrics + persist comparison row + confusion-matrix PNG."""
    y_pred = model.predict(X_test)

    metrics = {
        "model": name,
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
        metrics["roc_auc_ovr"] = roc_auc_score(y_test, proba, multi_class="ovr", average="macro")
    else:
        metrics["roc_auc_ovr"] = float("nan")

    # Append/replace row in comparison CSV.
    new_row = pd.DataFrame([metrics])
    if COMPARISON_CSV.exists():
        existing = pd.read_csv(COMPARISON_CSV)
        existing = existing[existing["model"] != name]
        out = pd.concat([existing, new_row], ignore_index=True)
    else:
        out = new_row
    out.to_csv(COMPARISON_CSV, index=False)
    log.info(f"Wrote '{name}' row to {COMPARISON_CSV.name} ({len(out)} model rows total)")

    # Confusion matrix.
    classes = label_encoder.classes_ if label_encoder is not None else np.unique(y_test)
    cm = confusion_matrix(y_test, y_pred, normalize="true")
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        cm,
        ax=ax,
        cmap="Blues",
        vmin=0,
        vmax=1,
        xticklabels=classes,
        yticklabels=classes,
        square=True,
        cbar_kws={"shrink": 0.7},
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion matrix (row-normalised) — {name}")
    plt.xticks(rotation=90, fontsize=7)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    cm_path = CONFUSION_DIR / f"{name}.png"
    fig.savefig(cm_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    log.info(f"Saved confusion matrix to {cm_path}")

    return metrics


# Run on the LogReg baseline.
metrics = evaluate_model(logreg_best, "logreg", X_test, y_test, label_encoder)
print(f"LogReg baseline:")
for k, v in metrics.items():
    fmt = f"{v:.4f}" if isinstance(v, float) else str(v)
    print(f"  {k:<18}: {fmt}")

## 7. RoBERTa Feasibility Spike — S1-T11

Embed 100 sample sentences through `roberta-base`, mean-pool, save `(100, 768)` to `data/embeddings/roberta_spike.npy`. Logs peak memory and wall-time so sprint 2 can size the full extraction. **Run on Colab GPU** — local CPU is too slow for the full version.

In [ ]:
# Implemented in S1-T11.

## 8. Groq LLM Smoke Test — S1-T13

Auths against Groq with `GROQ_API_KEY`, sends a one-line prompt, prints the response. Verifies the API path before the full LLM integration in sprint 2.

In [ ]:
# Implemented in S1-T13.

## 9. Minimal Gradio App — S1-T12

Two text inputs (headline, short description), one Predict button. Displays the LogReg baseline's predicted category + confidence. Launches a public share link from Colab. The full UI (LLM explanation + similar articles) ships in sprint 2.

In [ ]:
# Implemented in S1-T12.